# Introduction to DuckDB

## What is DuckDB?

DuckDB is an in-process SQL OLAP database management system. It's designed for analytical workloads and is optimized for fast queries on large datasets. Unlike traditional databases that run as separate servers, DuckDB runs embedded within your application.

## Key Features

- **Fast**: Optimized for analytical queries
- **Embedded**: No server setup required
- **SQL**: Standard SQL syntax
- **Python Integration**: Easy to use with pandas and other Python libraries
- **File-based**: Can work with CSV, Parquet, and other formats directly

## Why DuckDB for Analytics?

- Perfect for data exploration and prototyping
- Great for local development with dbt
- Can handle datasets that fit in memory
- Excellent performance for complex analytical queries

## Setting Up DuckDB

First, let's install and import DuckDB.

In [1]:
# Install DuckDB (if not already installed)
# !pip install duckdb

# Import required libraries
import duckdb
import polars as pl
import os

print("DuckDB version:", duckdb.__version__)
print("Polars version:", pl.__version__)

DuckDB version: 0.8.1
Polars version: 0.18.2


## Connecting to Our Database

Our dbt project uses a DuckDB database file. Let's connect to it and explore the structure.

In [2]:
# Connect to the DuckDB database
# The database file is created by our dbt project
db_path = "../my_db.duckdb"

if os.path.exists(db_path):
    con = duckdb.connect(db_path)
    print("Connected to DuckDB database successfully!")
else:
    print("Database file not found. Please run dbt first to create the database.")
    # For demo purposes, create an in-memory database
    con = duckdb.connect(":memory:")
    print("Connected to in-memory DuckDB database.")

Connected to DuckDB database successfully!


## Exploring the Database Schema

Let's see what tables are available in our database.

In [3]:
# List all tables in the database
tables_query = """
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'main'
ORDER BY table_name;
"""

tables = con.execute(tables_query).fetchall()
print("Tables in the database:")
for table in tables:
    print(f"- {table[0]}")

Tables in the database:
- customers
- orders
- products


## Basic Queries

Let's start with some basic SQL queries to explore our data.

In [4]:
# Query the customers table
customers_query = """
SELECT * 
FROM customers 
LIMIT 5;
"""

customers_df = con.execute(customers_query).fetchdf()
print("Sample customers:")
customers_df

: 

In [ ]:
# Query the products table
products_query = """
SELECT * 
FROM products 
LIMIT 5;
"""

products_df = con.execute(products_query).fetchdf()
print("Sample products:")
products_df

In [ ]:
# Query the orders table
orders_query = """
SELECT * 
FROM orders 
LIMIT 5;
"""

orders_df = con.execute(orders_query).fetchdf()
print("Sample orders:")
orders_df

## Filtering and Sorting

Let's practice filtering and sorting data.

In [ ]:
# Find customers from a specific location
location_query = """
SELECT customer_id, name, email
FROM customers
WHERE email LIKE '%gmail.com'
ORDER BY name
LIMIT 10;
"""

gmail_customers = con.execute(location_query).fetchdf()
print("Customers with Gmail addresses:")
gmail_customers

In [ ]:
# Find expensive products
expensive_products_query = """
SELECT product_id, name, price
FROM products
WHERE price > 50
ORDER BY price DESC;
"""

expensive_products = con.execute(expensive_products_query).fetchdf()
print("Expensive products (price > $50):")
expensive_products

## Joins

Now let's practice joining tables to get more meaningful insights.

In [ ]:
# Join orders with customers
orders_customers_query = """
SELECT 
    o.order_id,
    o.order_date,
    o.amount,
    c.name as customer_name,
    c.email
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_date DESC
LIMIT 10;
"""

orders_with_customers = con.execute(orders_customers_query).fetchdf()
print("Recent orders with customer information:")
orders_with_customers

In [ ]:
# Join orders with products (many-to-many through order_items if available)
# Since we might not have order_items, let's do a simple aggregation first
order_summary_query = """
SELECT 
    customer_id,
    COUNT(*) as total_orders,
    SUM(amount) as total_amount,
    AVG(amount) as avg_order_value,
    MAX(order_date) as last_order_date
FROM orders
GROUP BY customer_id
ORDER BY total_amount DESC
LIMIT 10;
"""

order_summary = con.execute(order_summary_query).fetchdf()
print("Top customers by total order amount:")
order_summary

## Aggregations and Analytics

Let's create some analytical queries with aggregations.

In [ ]:
# Monthly sales analysis
monthly_sales_query = """
SELECT 
    DATE_TRUNC('month', order_date) as month,
    COUNT(*) as orders_count,
    SUM(amount) as total_revenue,
    AVG(amount) as avg_order_value,
    COUNT(DISTINCT customer_id) as unique_customers
FROM orders
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY month;
"""

monthly_sales = con.execute(monthly_sales_query).fetchdf()
print("Monthly sales analysis:")
monthly_sales

In [ ]:
# Product performance analysis
product_performance_query = """
SELECT 
    p.product_id,
    p.name,
    p.price,
    COUNT(o.order_id) as times_ordered,
    SUM(o.amount) as total_revenue
FROM products p
LEFT JOIN orders o ON p.product_id = o.product_id  -- Assuming order_items table exists
GROUP BY p.product_id, p.name, p.price
ORDER BY total_revenue DESC
LIMIT 10;
"""

# Note: This assumes an order_items table. If not available, we'll skip this
try:
    product_performance = con.execute(product_performance_query).fetchdf()
    print("Product performance:")
    product_performance
except:
    print(
        "Product performance query requires order_items table (not available in this demo)"
    )

## Advanced DuckDB Features

DuckDB has many powerful features for analytics.

In [ ]:
# Using DuckDB's window functions
window_query = """
SELECT 
    customer_id,
    order_date,
    amount,
    SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date) as running_total,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) as order_number
FROM orders
ORDER BY customer_id, order_date
LIMIT 15;
"""

window_results = con.execute(window_query).fetchdf()
print("Window functions example (running totals by customer):")
window_results

In [ ]:
# Using DuckDB's approximate functions for large datasets
approx_query = """
SELECT 
    COUNT(*) as total_orders,
    APPROX_COUNT_DISTINCT(customer_id) as unique_customers,
    APPROX_QUANTILE(amount, 0.5) as median_order_value,
    APPROX_QUANTILE(amount, 0.95) as p95_order_value
FROM orders;
"""

approx_stats = con.execute(approx_query).fetchdf()
print("Approximate statistics:")
approx_stats

## Practice Exercises

Now it's your turn! Try these exercises to practice your DuckDB skills.

In [ ]:
# Exercise 1: Find the top 5 customers by number of orders
exercise1_query = """
-- Write your query here
SELECT customer_id, COUNT(*) as order_count
FROM orders
GROUP BY customer_id
ORDER BY order_count DESC
LIMIT 5;
"""

# Uncomment to run:
# result1 = con.execute(exercise1_query).fetchdf()
# result1

In [ ]:
# Exercise 2: Calculate daily order statistics for the last 30 days
exercise2_query = """
-- Write your query here
SELECT 
    DATE_TRUNC('day', order_date) as order_day,
    COUNT(*) as daily_orders,
    SUM(amount) as daily_revenue
FROM orders
WHERE order_date >= CURRENT_DATE - INTERVAL 30 DAY
GROUP BY DATE_TRUNC('day', order_date)
ORDER BY order_day;
"""

# Uncomment to run:
# result2 = con.execute(exercise2_query).fetchdf()
# result2

In [ ]:
# Exercise 3: Find customers who haven't ordered in the last 90 days
exercise3_query = """
-- Write your query here
WITH last_order_dates AS (
    SELECT 
        customer_id,
        MAX(order_date) as last_order_date
    FROM orders
    GROUP BY customer_id
)
SELECT 
    c.customer_id,
    c.name,
    c.email,
    l.last_order_date,
    CURRENT_DATE - l.last_order_date as days_since_last_order
FROM customers c
JOIN last_order_dates l ON c.customer_id = l.customer_id
WHERE l.last_order_date < CURRENT_DATE - INTERVAL 90 DAY
ORDER BY days_since_last_order DESC;
"""

# Uncomment to run:
# result3 = con.execute(exercise3_query).fetchdf()
# result3

## Working with Polars Integration

DuckDB integrates seamlessly with Polars for fast data analysis workflows.

In [ ]:
# Read data directly into Polars
customers_pl = con.execute("SELECT * FROM customers").pl()
orders_pl = con.execute("SELECT * FROM orders").pl()

print("Customers shape:", customers_pl.shape)
print("Orders shape:", orders_pl.shape)

# Perform analysis with Polars
merged_df = orders_pl.join(customers_pl, on="customer_id", how="left")
print("\nMerged data sample:")
print(merged_df.head())

In [ ]:
# Use DuckDB's Polars integration for complex queries
complex_query = """
SELECT 
    c.name,
    COUNT(o.order_id) as order_count,
    SUM(o.amount) as total_spent,
    AVG(o.amount) as avg_order_value
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
HAVING COUNT(o.order_id) > 0
ORDER BY total_spent DESC
LIMIT 10;
"""

top_customers = con.execute(complex_query).pl()
print("Top customers analysis:")
print(top_customers)

## Closing the Connection

Always remember to close your database connection when done.

In [ ]:
# Close the connection
con.close()
print("Database connection closed.")

## Key Takeaways

- DuckDB is perfect for analytical workloads and local development
- It integrates seamlessly with Python and Polars
- Standard SQL syntax with powerful extensions
- Great for prototyping before moving to production databases
- Excellent performance for complex analytical queries

## Next Steps

- Explore DuckDB's documentation: https://duckdb.org/
- Try the exercises above
- Experiment with different file formats (CSV, Parquet)
- Learn about DuckDB's advanced features like extensions
- Check out Polars documentation: https://pola.rs/